# 68 — Structured System Dynamics (SyD)

**Dataset:** Structured Retrospective (task-planning app; N=61 behavioral participants)
**Model family:** System Dynamics (SyD)
**Blueprint:** `outputs/reports/structured/structured_syd_blueprint.md` — read this first. Part 0.5 of
the blueprint explicitly checks this notebook is not redundant with NeT/StoP/DyS before building it:
NeT computed static transition-edge counts, StoP computed isolated daily flow-rate series, DyS computed
weekly composition + trend — none of them computed a running STOCK (currently-open task count over
time) or tested lagged relationships between backlog and subsequent behavior. That's what this notebook
adds.
**Output root:** `outputs/model_outputs/structured/system_dynamics/`

---

## The stock: Backlog_w

`Backlog_w` = count of non-censored tasks created on or before week `w` that are still open at week `w`
(not yet completed or deleted). Right-censored (`unresolved`) tasks simply remain in the stock through
the last observed week — the correct treatment, not a gap.

**Left-censoring caveat:** `Backlog_w` starts at 0 for each participant's own week 0, but 60.7% of all
tasks are left-censored (pre-existed the window) — so this stock is very likely an UNDERCOUNT of the
participant's true backlog, especially early on. Reported explicitly, not corrected.

## Feedback-loop tests (pooled, population-level -- same design split as InT's MI tests)

1. Productive loop: `completed_count_w` -> `created_count_{w+1}` (predicted positive)
2. Overload loop: `Backlog_w` -> `created_count_{w+1}` (predicted negative)
3. Balancing loop: `Backlog_w` -> `reschedule_count_w` (same week, predicted positive)


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
FEATURES = PROJECT_ROOT / "outputs" / "features" / "structured"
OUT = PROJECT_ROOT / "outputs" / "model_outputs" / "structured" / "system_dynamics"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

MIN_WEEKS = 4

print("Output folder:", OUT)

## 1. Load Data

In [ ]:
beh = pd.read_csv(CLEANED / "structured_behavioral_clean.csv")
beh["event_timestamp_utc"] = pd.to_datetime(beh["event_timestamp_utc"], format="mixed", utc=True)
tasks = pd.read_csv(FEATURES / "structured_task_lifecycle_features.csv")
pf = pd.read_csv(FEATURES / "structured_participant_features.csv")

print(f"Tasks: {len(tasks):,}  Participants: {pf.shape[0]}")

## 2. Local-Day Conversion and Participant-Relative Week Index (same convention as StoP/DyS)

In [ ]:
def to_local_date_series(df):
    out = pd.Series(index=df.index, dtype="object")
    for tz, grp in df.groupby("event_timezone"):
        try:
            out.loc[grp.index] = grp["event_timestamp_utc"].dt.tz_convert(tz).dt.date
        except Exception:
            out.loc[grp.index] = pd.NaT
    return out

created = beh[beh["event_type"] == "Task Created"].copy()
created["local_date"] = to_local_date_series(created)
created = created.dropna(subset=["local_date"])
created["local_date"] = pd.to_datetime(created["local_date"])

# 13 task_ids have 2 "Task Created" events ~0.5s apart (duplicate/double-fire logging artifact,
# verified -- not a genuine re-creation). Dedupe to the earliest event per task_id before anything
# downstream assumes a 1:1 task_id -> creation-event mapping.
n_before = len(created)
created = created.sort_values("event_timestamp_utc").drop_duplicates(subset="task_id", keep="first")
print(f"Deduped {n_before - len(created)} duplicate Task Created events (13 task_ids double-logged)")

first_creation = created.groupby("participant_id")["local_date"].transform("min")
created["entry_week"] = ((created["local_date"] - first_creation).dt.days // 7).astype(int)
entry_week_map = created.set_index("task_id")["entry_week"]  # Series only -- nc_tasks already has participant_id
first_creation_map = created.groupby("participant_id")["local_date"].min()

print(f"Non-censored tasks with entry_week: {len(entry_week_map):,}")

## 3. Determine Exit Week (completion or deletion) per Non-Censored Task

In [ ]:
def has_completed_tag(s):
    return pd.notna(s) and "task_completed" in s.split("|")

beh_upd = beh[beh["event_type"] == "Task Updated"].copy()
completion_events = beh_upd[beh_upd["update_type_list_str"].apply(has_completed_tag)].copy()
completion_events["local_date"] = to_local_date_series(completion_events)
completion_events = completion_events.dropna(subset=["local_date"]).sort_values("event_timestamp_utc")
last_completion_date = completion_events.groupby("task_id")["local_date"].last()

deleted_events = beh[beh["event_type"] == "Task Deleted"].copy()
deleted_events["local_date"] = to_local_date_series(deleted_events)
deleted_events = deleted_events.dropna(subset=["local_date"]).sort_values("event_timestamp_utc")
last_deletion_date = deleted_events.groupby("task_id")["local_date"].last()

resolution_date = last_completion_date.combine_first(last_deletion_date)

nc_tasks = tasks[tasks["has_creation_event"]].copy()
nc_tasks["entry_week"] = nc_tasks["task_id"].map(entry_week_map)
nc_tasks["resolution_date"] = nc_tasks["task_id"].map(resolution_date)
nc_tasks["first_creation_date"] = nc_tasks["participant_id"].map(first_creation_map)
nc_tasks["exit_week"] = np.where(
    nc_tasks["resolution_date"].notna(),
    ((pd.to_datetime(nc_tasks["resolution_date"]) - pd.to_datetime(nc_tasks["first_creation_date"])).dt.days // 7),
    np.nan
)

print(f"Non-censored tasks with determinable exit_week: {nc_tasks['exit_week'].notna().sum():,} / {len(nc_tasks):,}")
print(f"Still open (unresolved) at end of window: {nc_tasks['exit_week'].isna().sum():,}")

## 4. Build Weekly Backlog Stock + Flow Counts per Participant

In [ ]:
def has_reschedule_tag(s):
    return pd.notna(s) and bool({"date_changed", "start_time_changed"} & set(s.split("|")))

reschedule_events = beh_upd[beh_upd["update_type_list_str"].apply(has_reschedule_tag)].copy()
reschedule_events["local_date"] = to_local_date_series(reschedule_events)
reschedule_events = reschedule_events.dropna(subset=["local_date"])

records = []
all_pids = sorted(pf["participant_id"].unique())

for pid in all_pids:
    p_tasks = nc_tasks[nc_tasks["participant_id"] == pid]
    if len(p_tasks) == 0:
        continue
    max_week = int(p_tasks["entry_week"].max())

    p_created = created[created["participant_id"] == pid]
    created_per_week = p_created.groupby("entry_week").size()

    p_completed_ids = set(p_tasks[p_tasks["lifecycle_status"] == "completed"]["task_id"])
    comp_ev = completion_events[completion_events["task_id"].isin(p_completed_ids)].copy()
    fc_date = first_creation_map.get(pid)
    comp_ev["week"] = ((pd.to_datetime(comp_ev["local_date"]) - fc_date).dt.days // 7)
    completed_per_week = comp_ev.groupby("week").size()

    p_deleted_ids = set(p_tasks[p_tasks["lifecycle_status"] == "deleted"]["task_id"])
    del_ev = deleted_events[deleted_events["task_id"].isin(p_deleted_ids)].copy()
    del_ev["week"] = ((pd.to_datetime(del_ev["local_date"]) - fc_date).dt.days // 7)
    deleted_per_week = del_ev.groupby("week").size()

    p_resched = reschedule_events[reschedule_events["participant_id"] == pid].copy()
    p_resched["week"] = ((pd.to_datetime(p_resched["local_date"]) - fc_date).dt.days // 7)
    p_resched = p_resched[p_resched["week"] >= 0]
    resched_per_week = p_resched.groupby("week").size()

    for w in range(0, max_week + 1):
        backlog = int((
            (p_tasks["entry_week"] <= w) &
            ((p_tasks["exit_week"].isna()) | (p_tasks["exit_week"] > w))
        ).sum())
        records.append({
            "participant_id": pid, "week_idx": w, "Backlog_w": backlog,
            "created_count_w": int(created_per_week.get(w, 0)),
            "completed_count_w": int(completed_per_week.get(w, 0)),
            "deleted_count_w": int(deleted_per_week.get(w, 0)),
            "reschedule_count_w": int(resched_per_week.get(w, 0)),
        })

backlog_long = pd.DataFrame(records)
print(f"Backlog stock long table: {backlog_long.shape}")
backlog_long.head(10)

## 5. Save Backlog Stock Long Table

In [ ]:
backlog_long.to_csv(OUT / "backlog_stock_long.csv", index=False)
print(f"Saved: backlog_stock_long.csv  ({backlog_long.shape})")

## 6. Sanity Checks

In [ ]:
print("Backlog_w must be non-negative:", (backlog_long["Backlog_w"] >= 0).all())
print("created_count_w, completed_count_w, etc. must be non-negative:",
      (backlog_long[["created_count_w","completed_count_w","deleted_count_w","reschedule_count_w"]] >= 0).all().all())
print()
print("Backlog_w summary statistics:")
print(backlog_long["Backlog_w"].describe().round(2))

## 7. Per-Participant Summary (Part 3 of blueprint)

In [ ]:
weeks_per_p = backlog_long.groupby("participant_id")["week_idx"].nunique()

syd_records = []
for pid, grp in backlog_long.groupby("participant_id"):
    n_weeks = grp["week_idx"].nunique()
    is_eligible = n_weeks >= MIN_WEEKS
    rec = {"participant_id": pid, "n_weeks": n_weeks, "is_syd_eligible": is_eligible,
           "backlog_mean": round(float(grp["Backlog_w"].mean()), 3),
           "backlog_peak": int(grp["Backlog_w"].max())}
    if is_eligible:
        x = grp["week_idx"].values.astype(float)
        y = grp["Backlog_w"].values.astype(float)
        slope = float(np.polyfit(x, y, 1)[0]) if x.std() > 0 else np.nan
        rec["backlog_trend_slope"] = round(slope, 4)

        # Sensitivity check: trim the final week (immature-cohort right-censoring -- recently
        # created tasks haven't had time to resolve yet, which could mechanically inflate the
        # apparent backlog right at the end of the window and bias the slope upward).
        grp_trim = grp[grp["week_idx"] < grp["week_idx"].max()]
        if len(grp_trim) >= 3 and grp_trim["week_idx"].std() > 0:
            xt = grp_trim["week_idx"].values.astype(float)
            yt = grp_trim["Backlog_w"].values.astype(float)
            rec["backlog_trend_slope_trimmed"] = round(float(np.polyfit(xt, yt, 1)[0]), 4)
        else:
            rec["backlog_trend_slope_trimmed"] = np.nan
    else:
        rec["backlog_trend_slope"] = np.nan
        rec["backlog_trend_slope_trimmed"] = np.nan
    syd_records.append(rec)

syd_df = pd.DataFrame(syd_records)

# add back participants with zero non-censored tasks entirely (never appear in backlog_long)
all_pids_set = set(pf["participant_id"].unique())
missing = all_pids_set - set(syd_df["participant_id"])
if missing:
    fill = pd.DataFrame([{"participant_id": p, "n_weeks": 0, "is_syd_eligible": False,
                           "backlog_mean": np.nan, "backlog_peak": np.nan,
                           "backlog_trend_slope": np.nan, "backlog_trend_slope_trimmed": np.nan}
                          for p in missing])
    syd_df = pd.concat([syd_df, fill], ignore_index=True)

syd_df = syd_df.merge(pf[["participant_id", "pct_tasks_left_censored"]], on="participant_id", how="left")

elig = syd_df[syd_df["is_syd_eligible"]]
print(f"SyD-eligible: {len(elig)} / {len(syd_df)} ({len(elig)/len(syd_df)*100:.1f}%) -- out of all 61 participants")
print(elig[["backlog_mean", "backlog_peak", "backlog_trend_slope", "backlog_trend_slope_trimmed"]].describe().round(3))

## 7b. Right-Censoring Sensitivity Check

`backlog_trend_slope` was positive for essentially all eligible participants -- checked whether this is
a genuine finding or an artifact of immature-cohort right-censoring (tasks created near the end of the
window haven't had time to resolve yet, mechanically inflating the final week's backlog). Resolution
latency among tasks that DO resolve is short (median 0.66 days, 90th percentile ~5 days -- checked
separately), so this should only bias the single final week, not the whole trajectory. Comparing the raw
slope against a version with the final week trimmed out tests this directly.

In [ ]:
same_sign = (np.sign(elig["backlog_trend_slope"]) == np.sign(elig["backlog_trend_slope_trimmed"]))
print(f"Sign agreement between raw and trimmed slope: {same_sign.sum()}/{same_sign.notna().sum()} "
      f"({same_sign.mean()*100:.1f}%)")
print()
print("Raw vs trimmed slope, side by side:")
print(elig[["backlog_trend_slope", "backlog_trend_slope_trimmed"]].describe().round(3))
print()
print("If sign agreement is high, the positive trend is NOT primarily a right-censoring artifact of the")
print("final week -- it reflects a genuine structural accumulation, most plausibly driven by the 26.8%")
print("of non-censored tasks that never resolve within the window at all (a persistent floor), not by")
print("recently-created tasks simply not having had time to resolve yet.")

## 8. Dominant Loop Label (rule-based, descriptive)

In [ ]:
slope_terc = elig["backlog_trend_slope"].quantile([1/3, 2/3]).values

def classify(slope):
    if slope >= slope_terc[1]:
        return "overload-trending"
    if slope <= slope_terc[0]:
        return "productive-trending"
    return "balanced"

syd_df.loc[elig.index, "dominant_loop_label"] = elig["backlog_trend_slope"].apply(classify)
print(syd_df["dominant_loop_label"].value_counts(dropna=False))

## 9. Save Per-Participant Summary

In [ ]:
syd_df.to_csv(OUT / "participant_syd_summary.csv", index=False)
print(f"Saved: participant_syd_summary.csv  ({syd_df.shape})")

## 10. Feedback-Loop Tests (Part 2 of blueprint — pooled, population-level, predicted signs stated
in advance)

In [ ]:
# Build (week w, week w+1) paired table per participant for lagged tests
pairs = []
for pid, grp in backlog_long.sort_values("week_idx").groupby("participant_id"):
    grp = grp.set_index("week_idx")
    for w in grp.index:
        if (w + 1) in grp.index:
            pairs.append({
                "participant_id": pid, "week_idx": w,
                "completed_count_w": grp.loc[w, "completed_count_w"],
                "Backlog_w": grp.loc[w, "Backlog_w"],
                "reschedule_count_w": grp.loc[w, "reschedule_count_w"],
                "created_count_w1": grp.loc[w + 1, "created_count_w"],
            })
pairs_df = pd.DataFrame(pairs)
print(f"Lagged week-pairs available: {len(pairs_df):,}")

loop_records = []

r, p = stats.pearsonr(pairs_df["completed_count_w"], pairs_df["created_count_w1"])
loop_records.append({"loop_name": "productive", "X": "completed_count_w", "Y": "created_count_{w+1}",
                      "pearson_r": round(r, 4), "p_value": round(p, 6), "n": len(pairs_df),
                      "predicted_sign": "positive"})

r, p = stats.pearsonr(pairs_df["Backlog_w"], pairs_df["created_count_w1"])
loop_records.append({"loop_name": "overload", "X": "Backlog_w", "Y": "created_count_{w+1}",
                      "pearson_r": round(r, 4), "p_value": round(p, 6), "n": len(pairs_df),
                      "predicted_sign": "negative"})

r, p = stats.pearsonr(backlog_long["Backlog_w"], backlog_long["reschedule_count_w"])
loop_records.append({"loop_name": "balancing", "X": "Backlog_w", "Y": "reschedule_count_w (same week)",
                      "pearson_r": round(r, 4), "p_value": round(p, 6), "n": len(backlog_long),
                      "predicted_sign": "positive"})

loop_df = pd.DataFrame(loop_records)
print(loop_df.to_string())

## 10b. Within-Participant (Demeaned) Robustness Check

`pearson_r=0.9456` for the productive loop is suspiciously strong for a week-to-week effect -- almost
certainly inflated by a **between-participant confound**: some participants are simply high-volume users
every week, so `completed_count_w` and `created_count_{w+1}` are both high for the SAME reason (that
participant's general activity level), not because completing tasks in week `w` causes more creation in
week `w+1`. Subtracting each participant's own mean from each series before correlating (person-mean
centering, the standard fixed-effects panel technique for this exact confound) isolates the genuine
within-person week-to-week relationship.

In [ ]:
def demean_by_participant(df, col):
    return df[col] - df.groupby("participant_id")[col].transform("mean")

pairs_df["completed_w_dm"] = demean_by_participant(pairs_df, "completed_count_w")
pairs_df["backlog_w_dm"] = demean_by_participant(pairs_df, "Backlog_w")
pairs_df["created_w1_dm"] = demean_by_participant(pairs_df, "created_count_w1")

backlog_long["backlog_dm"] = demean_by_participant(backlog_long, "Backlog_w")
backlog_long["resched_dm"] = demean_by_participant(backlog_long, "reschedule_count_w")

dm_records = []
r, p = stats.pearsonr(pairs_df["completed_w_dm"], pairs_df["created_w1_dm"])
dm_records.append({"loop_name": "productive", "pearson_r_within": round(r, 4), "p_value": round(p, 6), "n": len(pairs_df)})

r, p = stats.pearsonr(pairs_df["backlog_w_dm"], pairs_df["created_w1_dm"])
dm_records.append({"loop_name": "overload", "pearson_r_within": round(r, 4), "p_value": round(p, 6), "n": len(pairs_df)})

r, p = stats.pearsonr(backlog_long["backlog_dm"], backlog_long["resched_dm"])
dm_records.append({"loop_name": "balancing", "pearson_r_within": round(r, 4), "p_value": round(p, 6), "n": len(backlog_long)})

dm_df = pd.DataFrame(dm_records).rename(columns={"p_value": "p_value_within"}).drop(columns=["n"])
loop_df = loop_df.merge(dm_df, on="loop_name")
print(loop_df[["loop_name", "pearson_r", "pearson_r_within", "predicted_sign"]].to_string())
print()
print("pearson_r = pooled (conflates between-participant volume differences with within-week timing)")
print("pearson_r_within = person-mean-centered (isolates genuine week-to-week coupling)")

## 11. Save Feedback-Loop Test Results

In [ ]:
loop_df.to_csv(OUT / "feedback_loop_tests.csv", index=False)
print(f"Saved: feedback_loop_tests.csv  ({loop_df.shape})")

## 12. Figure 1 — Backlog Trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
cmap = plt.get_cmap("tab20")
for i, (pid, g) in enumerate(backlog_long[backlog_long["participant_id"].isin(elig["participant_id"])].groupby("participant_id")):
    g = g.sort_values("week_idx")
    ax.plot(g["week_idx"], g["Backlog_w"], alpha=0.5, color=cmap(i % 20), linewidth=1.2)
ax.set_xlabel("Week (participant-relative)")
ax.set_ylabel("Backlog_w (open non-censored tasks)")
ax.set_title("Weekly backlog trajectories (SyD-eligible participants)")
plt.tight_layout()
plt.savefig(FIG / "fig1_syd_backlog_trajectories.png", dpi=150)
plt.show()
print("Saved fig1_syd_backlog_trajectories.png")

## 13. Figure 2 — Feedback Loop Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].scatter(pairs_df["completed_count_w"], pairs_df["created_count_w1"], alpha=0.4, s=20, color="#55A868")
axes[0].set_xlabel("completed_count_w"); axes[0].set_ylabel("created_count_{w+1}")
axes[0].set_title(f"Productive loop (r={loop_df.loc[0,'pearson_r']})")

axes[1].scatter(pairs_df["Backlog_w"], pairs_df["created_count_w1"], alpha=0.4, s=20, color="#C44E52")
axes[1].set_xlabel("Backlog_w"); axes[1].set_ylabel("created_count_{w+1}")
axes[1].set_title(f"Overload loop (r={loop_df.loc[1,'pearson_r']})")

axes[2].scatter(backlog_long["Backlog_w"], backlog_long["reschedule_count_w"], alpha=0.4, s=20, color="#4C72B0")
axes[2].set_xlabel("Backlog_w"); axes[2].set_ylabel("reschedule_count_w")
axes[2].set_title(f"Balancing loop (r={loop_df.loc[2,'pearson_r']})")

plt.tight_layout()
plt.savefig(FIG / "fig2_syd_feedback_loop_scatter.png", dpi=150)
plt.show()
print("Saved fig2_syd_feedback_loop_scatter.png")

## 14. model_run_summary.md

In [ ]:
summary = f"""# Structured SyD -- Model Run Summary

**Run date:** 2026-08-03
**Notebook:** notebooks/68_structured_system_dynamics.ipynb
**Blueprint:** outputs/reports/structured/structured_syd_blueprint.md

## Eligibility
- SyD-eligible (>= {MIN_WEEKS} weeks): {int(syd_df['is_syd_eligible'].sum())}/{len(syd_df)} ({syd_df['is_syd_eligible'].mean()*100:.1f}%)

## Backlog stock summary (eligible, N={len(elig)})
{elig[["backlog_mean","backlog_peak","backlog_trend_slope","backlog_trend_slope_trimmed"]].describe().round(3).to_string()}

## Right-censoring sensitivity check
Sign agreement between raw and final-week-trimmed backlog_trend_slope: {same_sign.sum()}/{same_sign.notna().sum()} ({same_sign.mean()*100:.1f}%)
-- high agreement means the positive trend is a genuine structural finding (driven by the 26.8%% of
non-censored tasks that never resolve within the window), not primarily a right-censoring artifact of
the final week alone.

## Dominant loop label distribution
{syd_df["dominant_loop_label"].value_counts(dropna=False).to_string()}

## Feedback-loop tests (pooled, population-level; predicted sign stated in advance)
{loop_df[["loop_name","X","Y","pearson_r","pearson_r_within","p_value","n","predicted_sign"]].to_string(index=False)}
-- pearson_r = pooled (conflates between-participant volume differences with within-week timing)
-- pearson_r_within = person-mean-centered, isolates genuine week-to-week coupling (see notebook 10b)

## Left-censoring caveat
Backlog_w is computed from non-censored tasks only and initialized at 0 per participant's own week 0 --
likely an UNDERCOUNT of true backlog, especially early in each participant's observed window, since
60.7%% of all tasks are left-censored (pre-existed the window).
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)